# Découpage du corpus en extraits (segments)

Ce notebook découpe les fichiers `.txt` du dossier `corpus_txt/` (voir le README) en
extraits de texte d'une longueur maximale choisie, sans jamais couper une phrase en
deux, puis répartit ces extraits en un **jeu d'annotation** et un **jeu d'inférence**
(mis de côté pour tester le classifieur une fois entraîné).

Le résultat est deux **fichiers CSV**, chacun avec deux colonnes :
- `text` : le texte de l'extrait ;
- `file_name` : le nom du fichier d'origine (contient déjà le journal et la date).

Ces fichiers peuvent ensuite être importés dans **ActiveTigger**.

Formats acceptés (au cas où vous ajouteriez d'autres documents) : `.docx`, `.odt`,
`.pdf`, `.md`, `.txt`.

## Comment utiliser ce notebook

1. Exécutez les cellules dans l'ordre jusqu'à la cellule **« Volume disponible »** :
   elle indique combien d'extraits ont été produits au total à partir de
   `corpus_txt/`. C'est cette information qui permet de choisir des tailles de jeux
   réalistes à l'étape suivante.
2. Dans la cellule **« Paramètres d'échantillonnage »**, ajustez au besoin
   `TAILLE_DATASET_ANNOTATION` et `TAILLE_DATASET_INFERENCE` (mettez `0` pour n'avoir
   qu'un seul fichier de sortie).
3. Exécutez le reste des cellules : les fichiers CSV sont créés à côté de ce notebook.

> ℹ️ Compatible Google Colab : importez le dossier `corpus_txt/` avant de lancer les
> cellules.


## 1. Installation et imports

In [ ]:
# Installation des dépendances nécessaires pour lire les différents formats de fichiers
%pip install -q python-docx odfpy pypdf pandas


In [ ]:
import os
import re
import glob
import random

import pandas as pd

from docx import Document as DocumentDocx
from odf.opendocument import load as load_odt
from odf.text import P as OdtParagraph
from odf import teletype
from pypdf import PdfReader

print("Imports OK")


## 2. Paramètres de découpage

C'est ici que vous réglez le comportement du découpage en extraits.


In [ ]:
# --- Dossier contenant les fichiers à découper ---
DOSSIER_DOCUMENTS = "corpus_txt"

# --- Longueur maximale d'un extrait, en nombre de caractères ---
# Le découpage respecte toujours les phrases : un extrait peut donc être un peu plus
# court que TAILLE_MAX, mais jamais coupé au milieu d'une phrase.
TAILLE_MAX = 500

# --- Longueur minimale d'un extrait à conserver ---
# Les tout derniers extraits d'un document, trop courts pour être utiles, sont ignorés.
TAILLE_MIN_GARDEE = 150

print(f"Dossier des documents : {DOSSIER_DOCUMENTS}")
print(f"Taille maximale d'un extrait : {TAILLE_MAX} caractères")


## 3. Lecture des documents

Une fonction de lecture différente est utilisée selon l'extension du fichier ; toutes
renvoient le texte intégral du document sous forme d'une seule chaîne de caractères.


In [ ]:
def lire_txt(chemin):
    """Lit un fichier .txt (essaie l'UTF-8, puis se rabat sur le Latin-1)."""
    try:
        with open(chemin, "r", encoding="utf-8") as f:
            return f.read()
    except UnicodeDecodeError:
        with open(chemin, "r", encoding="latin-1") as f:
            return f.read()


def lire_md(chemin):
    """Lit un fichier .md comme du texte brut (la syntaxe Markdown n'est pas retirée)."""
    return lire_txt(chemin)


def lire_docx(chemin):
    """Lit un fichier Word (.docx) en concaténant tous les paragraphes."""
    document = DocumentDocx(chemin)
    return "\n".join(p.text for p in document.paragraphs)


def lire_odt(chemin):
    """Lit un fichier OpenDocument Text (.odt) en concaténant tous les paragraphes."""
    document = load_odt(chemin)
    paragraphes = document.getElementsByType(OdtParagraph)
    return "\n".join(teletype.extractText(p) for p in paragraphes)


def lire_pdf(chemin):
    """Lit un fichier PDF en concaténant le texte extrait de chaque page."""
    lecteur = PdfReader(chemin)
    pages = [page.extract_text() or "" for page in lecteur.pages]
    return "\n".join(pages)


LECTEURS_PAR_EXTENSION = {
    ".txt": lire_txt,
    ".md": lire_md,
    ".docx": lire_docx,
    ".odt": lire_odt,
    ".pdf": lire_pdf,
}


def lire_document(chemin):
    """
    Lit un document quel que soit son format (parmi ceux supportés) et renvoie son
    texte intégral. Lève une ValueError si l'extension n'est pas supportée.
    """
    extension = os.path.splitext(chemin)[1].lower()
    lecteur = LECTEURS_PAR_EXTENSION.get(extension)
    if lecteur is None:
        raise ValueError(f"Extension non supportée : '{extension}'")
    return lecteur(chemin)


## 4. Découpage en phrases puis en extraits

Le texte est d'abord découpé en phrases (à partir de la ponctuation `.`, `!`, `?`,
`…`), puis les phrases sont regroupées les unes après les autres tant que la taille
de l'extrait ne dépasse pas `TAILLE_MAX`. Dès qu'ajouter la phrase suivante
dépasserait cette limite, l'extrait est terminé et un nouveau commence : **une
phrase n'est donc jamais coupée en deux**.

Cas particulier : si une phrase est à elle seule plus longue que `TAILLE_MAX`, elle
forme son propre extrait (plus long que `TAILLE_MAX`), car il est impossible de la
couper sans casser une phrase.


In [ ]:
# Regex de découpage en phrases : on coupe après un signe de ponctuation terminale
# suivi d'un espace, en gérant aussi les points de suspension.
RE_PHRASES = re.compile(r'(?<=[\.\!\?\u2026])\s+')


def decouper_en_phrases(texte):
    """Découpe un texte en phrases sans perdre la ponctuation."""
    texte = re.sub(r'\s+', ' ', texte).strip()
    if not texte:
        return []
    phrases = RE_PHRASES.split(texte)
    return [p.strip() for p in phrases if p.strip()]


def segmenter_texte(texte, taille_max=TAILLE_MAX, taille_min_gardee=TAILLE_MIN_GARDEE):
    """
    Segmente un texte en extraits d'au plus `taille_max` caractères, sans jamais
    couper une phrase. Les tout derniers extraits plus courts que
    `taille_min_gardee` sont écartés.
    """
    phrases = decouper_en_phrases(texte)
    if not phrases:
        return []

    segments = []
    segment_courant = ""

    for phrase in phrases:
        if not segment_courant:
            segment_courant = phrase
            continue

        candidat = segment_courant + " " + phrase

        if len(candidat) <= taille_max:
            segment_courant = candidat
        else:
            segments.append(segment_courant)
            segment_courant = phrase

    if segment_courant:
        segments.append(segment_courant)

    segments = [s for s in segments if len(s) >= taille_min_gardee]

    return segments


## 5. Traitement de tous les documents du dossier

On repère automatiquement, dans `DOSSIER_DOCUMENTS`, tous les fichiers dont
l'extension est supportée. Toute erreur de lecture sur un fichier (format
inattendu, fichier corrompu, etc.) est signalée mais n'interrompt pas le
traitement des autres fichiers.


In [ ]:
extensions_supportees = tuple(LECTEURS_PAR_EXTENSION.keys())

chemins_fichiers = sorted(
    chemin for chemin in glob.glob(os.path.join(DOSSIER_DOCUMENTS, "*"))
    if chemin.lower().endswith(extensions_supportees)
)

print(f"{len(chemins_fichiers)} fichier(s) trouvé(s) dans '{DOSSIER_DOCUMENTS}' :")
for chemin in chemins_fichiers:
    print(f"  - {os.path.basename(chemin)}")


In [ ]:
lignes = []  # chaque ligne : {"text": ..., "file_name": ...}

for chemin in chemins_fichiers:
    nom_fichier = os.path.basename(chemin)
    print(f"Traitement de '{nom_fichier}' ...")

    try:
        texte = lire_document(chemin)
    except Exception as e:
        print(f"  [ERREUR] Lecture impossible, fichier ignoré : {e}")
        continue

    if not texte or not texte.strip():
        print("  -> Fichier vide, ignoré.")
        continue

    try:
        segments = segmenter_texte(texte)
    except Exception as e:
        print(f"  [ERREUR] Découpage impossible, fichier ignoré : {e}")
        continue

    if not segments:
        print("  -> Aucun extrait exploitable après découpage, fichier ignoré.")
        continue

    for segment in segments:
        lignes.append({"text": segment, "file_name": nom_fichier})

    print(f"  -> {len(segments)} extrait(s) ajouté(s).")

print(f"\nTotal : {len(lignes)} extrait(s) produit(s) à partir de {len(chemins_fichiers)} fichier(s).")


## 6. Volume disponible

**Regardez ce nombre avant de choisir vos tailles de jeux à l'étape suivante** : il
n'est pas possible de demander plus d'extraits que ce total (annotation + inférence
confondus).


In [ ]:
print(f"Nombre total d'extraits disponibles avant échantillonnage : {len(lignes)}")


## 7. Paramètres d'échantillonnage

Réglez ici la taille du jeu à annoter et celle du jeu d'inférence, en vous basant sur
le nombre total affiché ci-dessus. Les deux jeux sont **disjoints** (aucun extrait en
commun) : le jeu d'inférence sert à tester le classifieur une fois entraîné sur des
extraits qu'il n'a jamais vus pendant l'annotation.

Mettez `TAILLE_DATASET_INFERENCE = 0` si vous ne voulez qu'un seul fichier de sortie.


In [ ]:
# --- Taille du jeu à annoter (entraînement du classifieur) ---
TAILLE_DATASET_ANNOTATION = 750

# --- Taille du jeu d'inférence, mis de côté pour tester le classifieur (0 = aucun) ---
TAILLE_DATASET_INFERENCE = 250

# --- Graine aléatoire, pour un tirage reproductible ---
SEED = 42

# --- Fichiers CSV de sortie ---
FICHIER_ANNOTATION = "corpus_annotation.csv"
FICHIER_INFERENCE = "corpus_inference.csv"

besoin_total = TAILLE_DATASET_ANNOTATION + TAILLE_DATASET_INFERENCE
if besoin_total > len(lignes):
    print(f"[AVERTISSEMENT] Vous demandez {besoin_total} extraits au total, mais seulement "
          f"{len(lignes)} sont disponibles. Réduisez TAILLE_DATASET_ANNOTATION et/ou "
          f"TAILLE_DATASET_INFERENCE, ou ajoutez des documents dans '{DOSSIER_DOCUMENTS}'.")
else:
    print(f"OK : {besoin_total} extraits demandés sur {len(lignes)} disponibles.")


## 8. Tirage et export des jeux de données

In [ ]:
random.seed(SEED)
lignes_melangees = lignes.copy()
random.shuffle(lignes_melangees)

lignes_annotation = lignes_melangees[:TAILLE_DATASET_ANNOTATION]
lignes_inference = lignes_melangees[TAILLE_DATASET_ANNOTATION:TAILLE_DATASET_ANNOTATION + TAILLE_DATASET_INFERENCE]

df_annotation = pd.DataFrame(lignes_annotation, columns=["text", "file_name"])
df_annotation.to_csv(FICHIER_ANNOTATION, index=False, encoding="utf-8")
print(f"Fichier '{FICHIER_ANNOTATION}' enregistré ({len(df_annotation)} lignes).")

if TAILLE_DATASET_INFERENCE > 0:
    df_inference = pd.DataFrame(lignes_inference, columns=["text", "file_name"])
    df_inference.to_csv(FICHIER_INFERENCE, index=False, encoding="utf-8")
    print(f"Fichier '{FICHIER_INFERENCE}' enregistré ({len(df_inference)} lignes).")

df_annotation.head()
